# 03 大模型应用开发 · 第01课：调用真实大模型API

> 上一阶段的聊天后端使用规则生成模拟回复。本课只替换最核心的一环：向真实模型发送一段文本并取得回答。流式响应、历史对话、RAG和Agent留到后续课程。

## 1. 本课完成标准

学完后你应能：

1. 区分ChatGPT网页产品、模型和API；
2. 解释API密钥的用途并避免把密钥提交到Git；
3. 看懂`OpenAI()`、`client.responses.create()`及其返回值；
4. 发出一次文本请求并读取`response.output_text`；
5. 把模型调用封装成可用假客户端离线测试的函数；
6. 根据认证、额度、模型名和网络错误确定排查方向。

## 2. 安装依赖与准备密钥

在本课程目录执行：

```powershell
python -m pip install -r requirements.txt
python -c "import openai, dotenv; print(openai.__version__)"
```

`python`启动当前解释器；`-m pip`用同一个解释器运行pip模块；`install`表示安装；`-r requirements.txt`表示从依赖清单读取包名。Notebook中也可运行`%pip install -r requirements.txt`，它会面向当前内核安装；安装后若仍无法导入，应重启内核。

依赖用途：`openai`是官方Python SDK；`python-dotenv`把本地`.env`加载进环境变量；`pytest`运行离线作业测试。复制`.env.example`为`.env`，再填入自己的密钥。`.env`不得上传GitHub。API使用可能产生费用，实际可用模型和额度由你的API账户决定，ChatGPT订阅不等同于API额度。

## 3. 网页聊天、模型与API不是同一个东西

- **ChatGPT网页或客户端**：给人使用的完整产品，界面、账号和产品功能由平台管理；
- **模型**：根据输入生成输出的能力，例如某个文本模型；
- **API**：程序按规定格式向服务发送请求的接口；
- **SDK**：把HTTP请求细节封装成Python类和方法的工具包。

程序流程是：Python代码 → SDK → HTTPS API请求 → 模型服务 → SDK响应对象 → 你的代码。SDK减少了手写请求头和JSON的工作，但请求仍然经过网络，并可能受到认证、额度、限流和网络状态影响。

## 4. API密钥与环境变量

API密钥相当于程序访问API的凭证，服务端用它识别账户和授权。不能直接写进Notebook或Python文件，因为Git历史和分享文件都可能泄露它。

`load_dotenv()`来自`python-dotenv`，查找`.env`并把其中键值载入当前Python进程的环境变量；返回布尔值表示是否找到文件。`os.getenv(name)`来自Python标准库`os`，读取名为`name`的环境变量；不存在时返回`None`。模型名不是秘密，但也放进环境变量，便于不改代码就切换模型。

正常情况：两项都有非空字符串。变化情况：系统环境变量可直接提供值，不一定必须有`.env`。错误情况：密钥缺失时应在发请求前停止，并给出清楚提示。

In [ ]:
# 练习1：只读取模型名，不要在这里填写或打印真实API密钥
import os

model_name = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")
# TODO：创建布尔变量 model_name_is_valid，判断model_name是否为非空字符串
model_name_is_valid = False

In [ ]:
try:
    _check_value = model_name_is_valid
    if not isinstance(_check_value, bool):
        print("❌ model_name_is_valid必须是布尔值。")
    elif _check_value != (isinstance(model_name, str) and bool(model_name.strip())):
        print("❌ 判断结果不符合‘非空字符串’规则。")
    else:
        print("✅ 模型名环境变量检查通过。")
except NameError:
    print("❌ 请先运行答题单元并创建model_name_is_valid。")

<details><summary>完成练习1后查看参考答案</summary>

`model_name_is_valid = isinstance(model_name, str) and bool(model_name.strip())`。先确认类型，再用`strip()`去除首尾空白并判断是否仍有内容。
</details>

## 5. 创建客户端并发出一次请求

`OpenAI`来自官方`openai`包，是一个类。`OpenAI()`调用类来创建客户端对象；未显式传`api_key`时，SDK默认读取`OPENAI_API_KEY`。把密钥交给环境变量比硬编码安全。

`client.responses.create(...)`调用Responses API：

- `model`是关键字参数，由开发者指定要使用的模型ID；
- `input`是关键字参数，提供本次用户输入；
- 返回值是SDK响应对象，不是普通字符串；
- `response.output_text`是SDK提供的便捷属性，用于汇总响应中的文本输出。

最小真实请求如下。只有你已经配置API密钥并愿意产生一次API调用时才运行：

```python
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
response = client.responses.create(
    model=os.getenv("OPENAI_MODEL", "gpt-5.6-luna"),
    input="请用一句话解释什么是API。",
)
print(response.output_text)
```

变化示例：改变`input`会得到新的请求。边界示例：空字符串虽然类型正确，但没有业务意义，应在自己的函数里提前拒绝。错误示例：密钥无效、账户无额度、模型无权限或网络不可达都会使方法抛出异常；不能把这些错误伪装成正常模型回答。

## 6. 课堂练习：组织请求参数

目标：理解`model`和`input`是谁提供的。创建字典`request_data`，只包含这两个键；把用户问题首尾空白去掉。正确结果不会访问网络。

In [ ]:
question = "  Python SDK解决什么问题？  "
selected_model = "demo-model"
request_data = {
    # TODO：填写两个键值
}

In [ ]:
try:
    _check_data = request_data
    if not isinstance(_check_data, dict):
        print("❌ request_data必须是字典。")
    elif set(_check_data) != {"model", "input"}:
        print("❌ 字典必须且只能包含model和input。")
    elif _check_data["model"] != selected_model or _check_data["input"] != question.strip():
        print("❌ 请检查模型值和去除首尾空白后的问题。")
    else:
        print("✅ 请求参数组织正确。")
except NameError:
    print("❌ 请先运行答题单元并创建request_data。")

<details><summary>完成练习2后查看参考答案</summary>

`request_data = {"model": selected_model, "input": question.strip()}`。字段名由Responses API定义，字段值由应用代码提供。
</details>

## 7. 为什么不能让自动测试真的调用模型

真实API调用依赖网络、密钥、额度、模型权限，并且输出不是每次逐字相同。这样的测试不稳定，还可能产生费用。更合理的方法是**依赖注入**：函数接收`client`参数，测试传入假客户端，生产代码传入`OpenAI()`客户端。

假客户端只记录`create`收到的参数并返回带`output_text`的简单对象。这样测试能精确检查：是否只调用一次、参数是否正确、输出是否被提取；它不会证明真实账户可用，真实连通性应单独手动验证。

In [ ]:
from types import SimpleNamespace

class PracticeResponses:
    def __init__(self):
        self.last_request = None

    def create(self, **kwargs):
        # TODO：保存kwargs到last_request，并返回output_text为‘离线回答’的对象
        pass

In [ ]:
try:
    _check_responses = PracticeResponses()
    _check_result = _check_responses.create(model="demo", input="你好")
    if _check_responses.last_request != {"model": "demo", "input": "你好"}:
        print("❌ create需要原样保存收到的关键字参数。")
    elif getattr(_check_result, "output_text", None) != "离线回答":
        print("❌ 返回对象需要包含output_text='离线回答'。")
    else:
        print("✅ 假Responses对象行为正确，全程未访问网络。")
except NameError:
    print("❌ 请先定义并运行PracticeResponses。")
except Exception as _check_error:
    print(f"❌ 假对象执行失败：{_check_error}")

<details><summary>完成练习3后查看参考答案</summary>

方法体可写为`self.last_request = kwargs`，然后`return SimpleNamespace(output_text="离线回答")`。`SimpleNamespace`来自标准库`types`，用于快速创建可通过属性访问数据的简单对象。
</details>

## 8. 错误发生在哪一层

| 现象 | 先检查 | 含义 |
|---|---|---|
| 创建客户端前就提示缺少密钥 | 环境变量或`.env` | 本地配置未准备好 |
| 认证失败 | API密钥 | 服务端不接受凭证 |
| 无权限或找不到模型 | 模型ID与账户权限 | 不能假设所有账户可用同一模型 |
| 额度或限流错误 | API账户额度、速率限制 | ChatGPT订阅不能替代API计费 |
| 连接超时 | 网络、代理和服务连通性 | 请求尚未正常完成 |
| 响应存在但没有可用文本 | 响应内容与程序假设 | 应明确报错，不能返回空字符串冒充成功 |

捕获异常的目的不是隐藏错误。用户界面可以显示友好信息，同时日志保留异常类型；密钥本身绝不能写入日志。

## 9. 课末作业

只修改`ai_reply.py`中的三个TODO，不修改测试。

1. `build_response_request(user_text, model)`：两项输入必须是去除首尾空白后的非空字符串；否则分别抛出含“用户输入”或“模型”的`ValueError`；成功返回只含`model`和`input`的字典。
2. `extract_output_text(response)`：读取`response.output_text`，去除首尾空白并返回；属性缺失或结果为空时抛出含“文本”的`ValueError`。
3. `generate_reply(client, user_text, model)`：构建请求，以关键字参数调用一次`client.responses.create`，再提取文本。函数不得读取环境变量或创建真实客户端。

运行：

```powershell
python -m pytest -q
```

应收集并通过10项测试。测试覆盖正常值、首尾空白、空输入、空模型、文本缺失与假客户端调用参数，全程不访问网络。

## 10. 理解题

1. ChatGPT网页产品、模型和API分别是什么？
2. 为什么不能把API密钥直接写进Notebook并提交Git？
3. `OpenAI()`创建的是什么对象？未传`api_key`时通常从哪里读取？
4. `client.responses.create(model=..., input=...)`的两个参数由谁提供，返回的为什么不是普通字符串？
5. `response.output_text`解决了什么问题？为什么仍应检查空文本？
6. 为什么pytest不应默认调用真实模型？假客户端能验证什么、不能验证什么？
7. 遇到认证失败、模型无权限、额度限制和网络超时时，排查方向有何不同？

## 11. 小结与官方资料

本课建立最小调用链：安全读取配置 → 创建SDK客户端 → 调用Responses API → 提取文本 → 用假客户端离线测试。下一步才会把这个函数接回现有FastAPI聊天后端。

官方资料：

- [OpenAI API模型目录](https://developers.openai.com/api/docs/models)
- [OpenAI模型与Responses API指导](https://developers.openai.com/api/docs/guides/latest-model)

模型、价格和账户权限会变化；课程中的模型名可通过`OPENAI_MODEL`调整，应以运行时账户和官方文档为准。